# Phase 6.3.3.1 — Simplified Decision Replay + Phase 6.4 Preflight
Zero new LLM calls. The original six-attempt archive and cache are read-only. Partial rankings remain partial; semantic applicability remains unverified. Extraction inputs are separated from template hypotheses. No specifications are extracted or generation authorized.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL='https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH='simplified-rag-v3'
ROOT=Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC=ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests/test_phase6_simplified_replay.py')],check=True)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
from tfl_agent.agent.phase6_simplified_replay import ARCHIVE_NAME, ARCHIVE_SHA, load_archive, run_replay
from tfl_agent.agent.phase6_specification_preflight import prepare_specification_inputs
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
FOLDER=BASE/'Phase633_Applicability_Pilot_Outputs'
ARCHIVE=FOLDER/ARCHIVE_NAME
if not ARCHIVE.is_file():
    matches=[]
    for p in FOLDER.glob('*.zip'):
        try:
            load_archive(p)
        except (ValueError,KeyError,OSError): continue
        matches.append(p)
    if len(matches)!=1:
        raise FileNotFoundError(f'Expected frozen archive {ARCHIVE}; matching alternatives: {matches}')
    ARCHIVE=matches[0]
load_archive(ARCHIVE)
print('Verified:',ARCHIVE,'| inner SHA256:',ARCHIVE_SHA)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

In [ ]:
from IPython.display import display
import pandas as pd
OUT=Path('/content/Phase6331_Simplified_Replay')/RUN_ID
summary,rows=run_replay(ARCHIVE,OUT/'Replay')
assert summary['new_llm_calls']==0 and len(rows)==6
assert not summary['generation_authorized']
print(json.dumps(summary,indent=2))
display(pd.DataFrame(rows)[['study_id','entry_id','expected','recommended_template','complete_ranking','protocol_valid','protocol_supported_template','issues']])
inputs,context=prepare_specification_inputs(ARCHIVE,OUT/'Phase64_Preflight')
assert len(inputs)==len(context)==6
assert all(set(r['payload'])=={'target','sap_evidence','allowed_fields'} for r in inputs)
assert all(r['confirmed_template'] is None for r in context)
print('Phase64 preflight:',len(inputs),'cases; specifications extracted: 0')

In [ ]:
DEST=BASE/'Phase6331_Simplified_Replay_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved:',saved)
print('Files:',sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()))

## Interpretation
This is retrospective replay under a smaller output contract, not a new LLM experiment. Raw prefix top-1 is distinct from full ranking top-1; unranked templates are not appended. Missing citation-level prose is not invented. A malformed citation remains blocked. Protocol-supported candidates are not semantically confirmed. Phase 6.4 preflight uses matching-oriented evidence only; field-specific retrieval and claim validation still need development. No API calls or DOCX generation.